# Class 5: Linear regression
DM587 · 2 November 2026

Run from top to bottom. Keep `odense-weather.csv` beside the notebook. Packages: NumPy, Matplotlib and pandas; SciPy supplies the triangular solver in the explicit QR example. The numerical examples match the slides. The notebook also includes an optional two-dimensional projection illustration; the slides use the book's plane diagram.

Reading: Boyd and Vandenberghe, *Introduction to Applied Linear Algebra*, §§2.3, 8.3, 10.4, 11.3, 11.5, 12.1–12.3 and 13.1–13.3. The explanations and figures here are original teaching examples following that notation. Housing values and the noisy polynomial example are illustrative. The function-approximation example uses actual values of cos(x), not measurements. Weather is the real Odense ERA5 extract used in Class 4.

No statistics background is assumed. We choose a family of functions, write their predictions as a matrix-vector product, and fit the coefficients by least squares.

This is a standard Jupyter notebook (.ipynb), suitable for Jupyter Notebook/JupyterLab and PyCharm with notebook support. Select a Python environment with the packages listed above and run cells in order. Figures are ordinary Matplotlib outputs; no embedded PDF viewer is required. Download the ZIP to keep the notebook and weather CSV together.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
plt.rcParams.update({"font.size": 12, "figure.figsize": (8.8, 3.6),
                     "axes.spines.top": False, "axes.spines.right": False})
figures = {}
checks = {}
metrics = {}
def finish(fig, ax, title, xlabel, ylabel):
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.18)
    fig.tight_layout()
def save_figure(name, fig):
    figures[name] = fig
    plt.show()


## Opening question: what function would you choose?
Look only at the dots in panels A, B and C. Suggest a simple family of functions for each panel. Must the curve pass through every point? How would you decide between several possible fits?

For each panel we sample a known underlying function at nine distinct inputs, then add independent simulated noise with standard deviation 0.12 (fixed seed 587). We hide the underlying functions at first. The first figure deliberately contains no fitted curves. Run the next cell only after making a prediction.

In [ ]:
opening_x = np.linspace(-1, 1, 9)
opening_grid = np.linspace(-1.2, 1.2, 500)
opening_rng = np.random.RandomState(587)
opening_true = [lambda v: 0.2 + 0.7 * v,
                lambda v: 0.3 - 0.9 * v ** 2,
                lambda v: 0.75 * np.sin(np.pi * v)]
opening_y = [function(opening_x) + 0.12 * opening_rng.randn(len(opening_x))
             for function in opening_true]
def opening_panels():
    fig, axs = plt.subplots(1, 3, figsize=(11, 3.3), sharey=True)
    for ax, values, title in zip(axs, opening_y, ["A", "B", "C"]):
        ax.scatter(opening_x, values, color="#203040", s=32, zorder=5)
        ax.set_title(title)
        ax.set_xlabel("Input x")
        ax.set_xlim(-1.2, 1.2)
        ax.set_ylim(-1.3, 1.3)
        ax.grid(alpha=0.18)
    axs[0].set_ylabel("Observed outcome y")
    fig.tight_layout()
    return fig, axs
fig, axs = opening_panels()
save_figure("opening-points", fig)


### Reveal: familiar candidates
A straight line, a quadratic and a sine-shaped curve are plausible low-complexity choices. Fit their coefficients by least squares. The sine frequency is fixed here at pi: we fit a constant, sine coefficient and cosine coefficient, not an unknown frequency. The line and quadratic generally do not pass through every perturbed observation.

The observed points are true function values plus simulated noise. The solid curve is fitted to these noisy observations; it is not simply the known generating function drawn again.

In [ ]:
opening_bases = [lambda v: np.column_stack([np.ones(len(v)), v]),
                 lambda v: np.column_stack([np.ones(len(v)), v, v ** 2]),
                 lambda v: np.column_stack([np.ones(len(v)), np.sin(np.pi * v), np.cos(np.pi * v)])]
opening_candidates = []
for values, basis in zip(opening_y, opening_bases):
    coefficients = np.linalg.lstsq(basis(opening_x), values, rcond=None)[0]
    opening_candidates.append(np.dot(basis(opening_grid), coefficients))
fig, axs = opening_panels()
for ax, curve, name in zip(axs, opening_candidates, ["Straight line", "Quadratic", "Sine-shaped"]):
    ax.plot(opening_grid, curve, color="#006D77", linewidth=2, label=name)
    ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
save_figure("opening-candidates", fig)


### Another reveal: exact interpolation
For N distinct scalar inputs there is one polynomial of degree at most N-1 that passes through every input–outcome pair. Here N=9, so a polynomial of degree at most 8 can interpolate every panel, including the sine-shaped data.

This does not make the polynomial the true or best predictive model. It also fits the small measurement perturbations, and its behaviour between or beyond the observed inputs can be very different. The grey edge regions in the next figure are outside the observed input interval.

How to choose? Consider the scientific shape you expect, performance on new/held-out observations, and complexity. Exact fit is one property, not the whole decision. Later slides make this explicit with separate training, validation and test data.

In [ ]:
opening_P = np.column_stack([opening_x ** j for j in range(9)])
opening_new_P = np.column_stack([opening_grid ** j for j in range(9)])
opening_interpolation_errors = []
fig, axs = opening_panels()
for ax, values, simple in zip(axs, opening_y, opening_candidates):
    polynomial = np.linalg.solve(opening_P, values)
    opening_interpolation_errors.append(float(np.max(np.abs(np.dot(opening_P, polynomial) - values))))
    ax.axvspan(-1.2, -1, color="#E9EDEF")
    ax.axvspan(1, 1.2, color="#E9EDEF")
    ax.plot(opening_grid, simple, color="#006D77", linewidth=1.7, label="Simple candidate")
    ax.plot(opening_grid, np.dot(opening_new_P, polynomial), color="#D56A35", linewidth=1.7,
            linestyle="--", label="Interpolating polynomial")
    ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
save_figure("opening-interpolation", fig)
print("Largest interpolation errors at observed points:", opening_interpolation_errors)
checks["opening_interpolation"] = bool(max(opening_interpolation_errors) < 1e-10)
checks["opening_distinct_inputs"] = bool(len(np.unique(opening_x)) == len(opening_x))
metrics["opening"] = {"observations_per_panel": len(opening_x), "degree_bound": 8,
                      "interpolation_errors": opening_interpolation_errors}


### Audience question: which is the better fit?
Compare the quadratic model with the interpolating polynomial on exactly the same noisy observations from panel B. Both panels use identical axes. Which would you choose, and what would you need to know before deciding? Do not select a winner solely because one curve passes through every point.

In [ ]:
quadratic_values = opening_y[1]
quadratic_curve = opening_candidates[1]
interpolating_coefficients = np.linalg.solve(opening_P, quadratic_values)
interpolating_curve = np.dot(opening_new_P, interpolating_coefficients)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True, sharey=True)
for ax, curve, title, colour in zip(axs,
        [quadratic_curve, interpolating_curve],
        ["Quadratic fit", "Interpolating polynomial (degree 8)"],
        ["#006D77", "#D56A35"]):
    ax.axvspan(-1.2, -1, color="#E9EDEF")
    ax.axvspan(1, 1.2, color="#E9EDEF")
    ax.scatter(opening_x, quadratic_values, color="#203040", s=40, zorder=5)
    ax.plot(opening_grid, curve, color=colour, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Input x")
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.18)
axs[0].set_ylabel("Observed outcome y")
fig.tight_layout()
save_figure("opening-choice", fig)


## 1. Models and observations
A model maps an input to a predicted outcome: $\hat y=\hat f(x)$. A straight-line model is $\hat f(x)=\theta_1+\theta_2x$. The superscript in $x^{(i)}$ means observation number, not a power. The residual is observed minus predicted, $r^{(i)}=y^{(i)}-\hat y^{(i)}$.

In the illustrative housing example, $x$ is area in units of 100 square metres and $y$ is price in millions of DKK. These examples demonstrate fitting; the coefficients are not causal effects.

In [ ]:
house_x = np.array([0.6, 0.8, 1.0, 1.3, 1.6, 2.0, 2.4])
house_y = np.array([1.10, 1.00, 1.42, 1.52, 1.95, 2.00, 2.62])
house_A = np.column_stack([np.ones(len(house_x)), house_x])
house_theta = np.linalg.lstsq(house_A, house_y, rcond=None)[0]
curve_x = np.linspace(0, 3, 200)
cal_x = np.linspace(-2, 2, 9)
cal_y = np.cos(cal_x)
cal_grid = np.linspace(-2, 2, 200)
fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.7))
axs[0].scatter(house_x, house_y, color="#203040")
axs[0].plot(curve_x, house_theta[0] + house_theta[1] * curve_x, color="#006D77", label="Straight line")
axs[0].set_title("Housing: area and price")
axs[0].set_xlabel("Area (100 square metres)"); axs[0].set_ylabel("Price (million DKK)")
axs[1].scatter(cal_x, cal_y, color="#203040")
for degree, label, colour in [(1, "Straight line", "#D56A35"), (2, "Quadratic", "#006D77")]:
    coefficients = np.polynomial.polynomial.polyfit(cal_x, cal_y, degree)
    axs[1].plot(cal_grid, np.polynomial.polynomial.polyval(cal_grid, coefficients), color=colour, label=label)
axs[1].set_title("Mathematics: approximate cos(x)")
axs[1].set_xlabel("x"); axs[1].set_ylabel("Function value")
for ax in axs: ax.legend(fontsize=9); ax.grid(alpha=0.18)
fig.suptitle("Illustrative houses and sampled function values", fontsize=12)
fig.tight_layout(); save_figure("motivation", fig)
fig, ax = plt.subplots()
ax.scatter(house_x, house_y, color="#203040", label="Observed houses")
ax.plot(curve_x, house_theta[0] + house_theta[1] * curve_x, color="#006D77", label="Fitted line")
ax.scatter([1.5], [house_theta[0] + 1.5 * house_theta[1]], marker="*", s=160, color="#D56A35", label="Prediction at 150 square metres")
ax.legend(fontsize=9)
finish(fig, ax, "Illustrative houses: a rule for a new input", "Area (100 square metres)", "Price (million DKK)")
save_figure("housing", fig)
print("Illustrative housing coefficients:", house_theta)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.3))
for intercept in [0, 0.7, 1.4]: axs[0].plot(curve_x, intercept + 0.8 * curve_x, label="Intercept = " + str(intercept))
for slope in [0.3, 0.8, 1.3]: axs[1].plot(curve_x, 0.7 + slope * curve_x, label="Slope = " + str(slope))
for ax, title in zip(axs, ["Change the intercept", "Change the slope"]):
    ax.set_title(title); ax.set_xlabel("x"); ax.set_ylabel("Predicted y"); ax.legend(fontsize=9); ax.grid(alpha=0.18)
fig.tight_layout(); save_figure("coefficients", fig)


### Can one line fit all three points?
The observations are (1, 1), (2, 2) and (3, 2). A line through the first two is y=x, but it predicts 3 at x=3 instead of the observed 2. The graph and the three equations describe the same inconsistent exact-fitting problem.

In [ ]:
exact_x = np.array([1., 2., 3.])
exact_y = np.array([1., 2., 2.])
fig, ax = plt.subplots(figsize=(4.7, 4.1))
line_x = np.linspace(0.6, 3.4, 100)
ax.plot(line_x, line_x, color="#006D77", label="Line through first two points")
ax.scatter(exact_x, exact_y, s=65, color="#203040", zorder=5, label="Observed points")
ax.scatter([3], [3], marker="x", s=85, color="#D56A35", zorder=6)
ax.plot([3, 3], [2, 3], "--", color="#D56A35", linewidth=1.8)
for xx, yy in zip(exact_x, exact_y):
    ax.annotate("(" + str(int(xx)) + ", " + str(int(yy)) + ")", (xx, yy),
                xytext=(-32, -20), textcoords="offset points", fontsize=10)
ax.annotate("Prediction: 3", (3, 3), xytext=(-85, 13), textcoords="offset points",
            color="#D56A35", fontsize=10)
ax.set_xlim(0.5, 3.6); ax.set_ylim(0.5, 3.6)
ax.set_xticks([1, 2, 3]); ax.set_yticks([1, 2, 3])
ax.set_xlabel("Input x"); ax.set_ylabel("Outcome y")
ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=0.18)
fig.tight_layout(); save_figure("exact-fitting", fig)
checks["exact_fitting_inconsistent"] = bool(not np.allclose(exact_x, exact_y))


## 2. Exact linear systems and rank
A row of $A\theta=b$ is one equation. The number of equations alone does not decide existence or uniqueness. Consistency means $b$ belongs to the column space of $A$; uniqueness, when consistent, means the nullspace contains only zero. Full column rank means all columns are independent.

The plot below lives in **parameter space**: every point is a candidate pair of coefficients. Each line is one equation, unlike the data plot above, where points were observations.

**From linear algebra to computation.** Existence and uniqueness are mathematical questions. Computing an answer also requires an algorithm, a work/memory budget and attention to rounding. Elimination and factorisation reduce a general system to triangular solves.

Although the inverse formula is correct, `solve(M, b)` need not form the inverse. Computing the entire inverse supplies solutions to all columns of the identity, then requires a further multiplication by `b`. For one right-hand side this does extra work and stores an extra square matrix. Both dense algorithms have cubic leading order; the advantage is avoiding unnecessary computation, not changing the exponent. The optional repeated-right-hand-side example is at the end.

### A larger elimination example
After the two-dimensional geometry picture, the numerical example uses three equations with coefficient rows (1,1,1), (2,3,1), (1,2,3) and right-hand side (6,11,14). First subtract twice row 1 from row 2, and row 1 from row 3. Then subtract the new row 2 from row 3. This leaves upper-triangular rows (1,1,1 | 6), (0,1,-1 | -1), (0,0,3 | 9).

The point of elimination is to arrange dependencies: the last equation has one unknown, and each earlier equation introduces one new unknown once the later answers are known. Back substitution gives theta_3=3, theta_2=2, theta_1=1. The diagonal must be nonzero for this direct back-substitution recipe. No row exchanges are needed in this particular example; general solvers use pivoting. We stop at triangular form rather than reducing the matrix to the identity. The code makes each row operation explicit before calling the general-purpose solver.

### Square nonsingular, square singular, and rectangular
A square matrix is nonsingular when its columns are independent. It has an inverse and gives one exact solution for every right-hand-side vector. A square matrix with dependent columns is singular: it has no inverse and may have infinitely many exact solutions or no exact solution, depending on the right-hand side.

For a rectangular regression matrix, use shape and rank rather than treating the matrix as invertible. The original equations may be inconsistent even with independent columns. Least squares gives unique coefficients if the columns are independent; with dependent columns, the fitted predictions remain unique but the coefficients are not. NumPy lstsq chooses a minimum-norm coefficient vector. We fit one outcome vector in this lecture; multiple right-hand sides are an optional extension at the end.

In [ ]:
S = np.array([[1., 1.], [2., 2.]])
print("Rank of the singular matrix:", np.linalg.matrix_rank(S))
for right_side in [np.array([2., 4.]), np.array([2., 5.])]:
    coefficients = np.linalg.lstsq(S, right_side, rcond=None)[0]
    print("Right-hand side:", right_side)
    print("Minimum-norm least-squares coefficients:", coefficients)
    print("Predictions:", np.dot(S, coefficients))
    print("Exact equations satisfied:", np.allclose(np.dot(S, coefficients), right_side))
checks["singular_consistent"] = bool(np.allclose(np.dot(S, [1., 1.]), [2., 4.]))
checks["singular_inconsistent_fit"] = bool(np.allclose(np.linalg.lstsq(S, [2., 5.], rcond=None)[0], [1.2, 1.2]))


### Counting arithmetic work
A dot product of length n needs n multiplications and n-1 additions: 2n-1 arithmetic operations. O(n) describes linear growth, ignoring constant factors and lower-order terms. Here we count real scalar additions/multiplications in standard dense algorithms, not Python statements or elapsed seconds.

An m by n matrix times an n-vector makes m row dot products: m(2n-1) operations, O(mn). A product of an m by n matrix and an n by p matrix makes mp dot products of length n: mp(2n-1), O(mnp). For square n by n matrices these become O(n²) and O(n³). Faster matrix-multiplication algorithms exist, but this is the standard row-by-column count used here.

The coloured examples below use the same A and x. Each output entry is one dot product. Efficient libraries can reduce overhead and exploit hardware without changing these standard arithmetic counts.

In [ ]:
work_A = np.arange(1, 13).reshape(4, 3)
work_x = np.array([1, 2, 3])
work_B = np.array([[1, 0], [2, 1], [3, 2]])
work_y = np.dot(work_A, work_x)
work_C = np.dot(work_A, work_B)
print("A times x:", work_y)
print("A times B:"); print(work_C)
print("Length-3 dot product: 3 multiplications + 2 additions =", 2 * 3 - 1)
print("4 by 3 matrix-vector product:", 4 * (2 * 3 - 1), "operations")
print("4 by 3 times 3 by 2 matrix product:", 4 * 2 * (2 * 3 - 1), "operations")
checks["work_dot"] = bool(np.dot(work_A[1], work_x) == 32)
checks["work_products"] = bool(work_y[1] == 32 and work_C[1, 1] == 17)
# Grid diagrams show exactly which entries form one output.
def work_grid(fig, rect, values, title, highlights):
    from matplotlib.patches import Rectangle
    values = np.asarray(values)
    if values.ndim == 1:
        values = values.reshape(-1, 1)
    rows, cols = values.shape
    ax = fig.add_axes(rect)
    for row in range(rows):
        for col in range(cols):
            colour = highlights.get((row, col), "#F3F6F8")
            ax.add_patch(Rectangle((col, rows - row - 1), 1, 1,
                                  facecolor=colour, edgecolor="white", linewidth=2))
            ax.text(col + 0.5, rows - row - 0.5, str(values[row, col]),
                    ha="center", va="center", fontsize=17, color="#203040")
    ax.set_xlim(0, cols); ax.set_ylim(0, rows)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title, fontsize=12, pad=10)
    return ax
fig, ax = plt.subplots(figsize=(9, 3))
colours = ["#006D77", "#D56A35", "#3F5C9A"]
for j, (v, w, colour) in enumerate(zip([4, 5, 6], [1, 2, 3], colours)):
    centre = 1 + 2 * j
    ax.text(centre, 2.5, str(v), ha="center", fontsize=23, color=colour)
    ax.text(centre, 1.6, "× " + str(w), ha="center", fontsize=23, color=colour)
    ax.annotate("", xy=(centre, 0.75), xytext=(centre, 1.3),
                arrowprops={"arrowstyle":"->", "color":colour})
    ax.text(centre, 0.35, str(v * w), ha="center", fontsize=23, color=colour)
    if j < 2: ax.text(centre + 1, 0.35, "+", ha="center", fontsize=23)
ax.text(6.6, 0.35, "= 32", fontsize=23, color="#203040")
ax.text(-0.3, 2.5, "Row a", fontsize=12, ha="right")
ax.text(-0.3, 1.6, "Vector x", fontsize=12, ha="right")
ax.set_xlim(-1.6, 8.4); ax.set_ylim(0, 3.1); ax.axis("off")
fig.tight_layout(); save_figure("work-dot", fig)
fig = plt.figure(figsize=(10, 3.2))
work_grid(fig, [0.02, 0.08, 0.33, 0.76], work_A, "A: 4 rows × 3 columns",
          {(1, j):"#B8DDE0" for j in range(3)})
work_grid(fig, [0.42, 0.08, 0.12, 0.76], work_x, "x: length 3",
          {(i, 0):"#B8DDE0" for i in range(3)})
work_grid(fig, [0.72, 0.08, 0.15, 0.76], work_y, "Ax: length 4", {(1, 0):"#F5C5A7"})
fig.text(0.37, 0.45, "×", fontsize=26); fig.text(0.62, 0.45, "=", fontsize=26)
save_figure("work-matvec", fig)
fig = plt.figure(figsize=(10, 3.2))
work_grid(fig, [0.01, 0.08, 0.30, 0.76], work_A, "A: 4 × 3",
          {(1, j):"#B8DDE0" for j in range(3)})
work_grid(fig, [0.40, 0.08, 0.22, 0.76], work_B, "B: 3 × 2",
          {(i, 1):"#B8DDE0" for i in range(3)})
work_grid(fig, [0.75, 0.08, 0.24, 0.76], work_C, "AB: 4 × 2", {(1, 1):"#F5C5A7"})
fig.text(0.33, 0.45, "×", fontsize=26); fig.text(0.67, 0.45, "=", fontsize=26)
save_figure("work-matmat", fig)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
sizes = np.arange(1, 9)
for power, label, colour in [(1,"Linear: n", "#006D77"),
                             (2,"Quadratic: n squared", "#D56A35"),
                             (3,"Cubic: n cubed", "#3F5C9A")]:
    axs[0].plot(sizes, sizes ** power, "o-", label=label, color=colour)
axs[0].set_xlabel("Problem size n"); axs[0].set_ylabel("Illustrative work units")
axs[0].set_title("Different growth rates"); axs[0].legend(fontsize=8)
axs[1].bar([0,1,2], [2,4,8], color=["#006D77","#D56A35","#3F5C9A"])
axs[1].set_xticks([0,1,2]); axs[1].set_xticklabels(["O(n)","O(n²)","O(n³)"])
axs[1].set_ylabel("Approximate work multiplier")
axs[1].set_title("When n doubles")
for i, number in enumerate([2,4,8]): axs[1].text(i, number+0.15, str(number)+"×", ha="center")
axs[1].set_ylim(0,9)
for ax in axs: ax.grid(axis="y", alpha=0.18); ax.set_axisbelow(True)
fig.tight_layout(); save_figure("work-growth", fig)


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 3.1))
u = np.linspace(-1, 4, 150)
axs[0].plot(u, 4 - 2 * u, label="2 theta_1 + theta_2 = 4")
axs[0].plot(u, (7 - u) / 3, label="theta_1 + 3 theta_2 = 7")
axs[0].scatter([1], [2], color="#203040", zorder=4); axs[0].set_title("Two independent equations")
axs[1].plot(u, 2 - u, label="theta_1 + theta_2 = 2")
axs[1].set_title("One equation: many solutions")
for v in [0, 1, 2]: axs[1].scatter([v], [2-v], color="#203040")
axs[2].plot(u, 4 - 2 * u); axs[2].plot(u, (7 - u) / 3); axs[2].plot(u, 1 - u)
axs[2].set_title("Three equations: no common point")
for ax in axs:
    ax.set_xlim(-1, 4); ax.set_ylim(-2, 5); ax.set_xlabel("theta_1"); ax.set_ylabel("theta_2"); ax.grid(alpha=0.18)
fig.tight_layout(); save_figure("systems", fig)
M = np.array([[1., 1., 1.], [2., 3., 1.], [1., 2., 3.]])
b = np.array([6., 11., 14.])
augmented = np.column_stack([M, b])
first_stage = augmented.copy()
first_stage[1] = first_stage[1] - 2 * first_stage[0]
first_stage[2] = first_stage[2] - first_stage[0]
triangular = first_stage.copy()
triangular[2] = triangular[2] - triangular[1]
print("After eliminating the first unknown from the lower rows:")
print(first_stage)
print("Upper-triangular augmented system:")
print(triangular)
answer3 = triangular[2, 3] / triangular[2, 2]
answer2 = (triangular[1, 3] - triangular[1, 2] * answer3) / triangular[1, 1]
answer1 = (triangular[0, 3] - triangular[0, 1] * answer2 - triangular[0, 2] * answer3) / triangular[0, 0]
back_substitution = np.array([answer1, answer2, answer3])
print("Back-substitution solution:", back_substitution)
checks["elimination_stages"] = bool(np.allclose(triangular, [[1, 1, 1, 6], [0, 1, -1, -1], [0, 0, 3, 9]]))
checks["back_substitution"] = bool(np.allclose(np.dot(M, back_substitution), b))
solution = np.linalg.solve(M, b)
print("Square system solution:", solution)
print("Check M times solution:", np.dot(M, solution))
checks["square_solve"] = bool(np.allclose(np.dot(M, solution), b))


### Counting elimination and substitution
Counting zeros is not counting scalar arithmetic operations. At pivot k, elimination updates (p-k) rows, each with (p-k) remaining coefficient entries. The inner update A[i,j] = A[i,j] - m*A[k,j] performs a multiplication and subtraction, so its main work is 2 times the sum of (p-k)^2: cubic in p. Computing multipliers and updating the right-hand side add only quadratic work.

Back substitution has just two loops. Across its rows it uses 0+1+...+(p-1) previously known values, with one multiplication and subtraction per value and one division per row. This totals p^2 scalar arithmetic operations. These are work counts, not Python execution timings. The simple pseudocode assumes nonzero pivots and omits row swaps.

In [ ]:
def elimination_entry_updates(size):
    updates = 0
    for k in range(size - 1):
        for i in range(k + 1, size):
            for j in range(k + 1, size):
                updates += 1
    return updates

def substitution_pairs(size):
    pairs = 0
    for i in range(size - 1, -1, -1):
        for j in range(i + 1, size):
            pairs += 1
    return pairs

for size in [5, 10, 20, 40]:
    updates = elimination_entry_updates(size)
    pairs = substitution_pairs(size)
    print("Matrix size:", size)
    print("Elimination multiply-subtract pairs:", updates)
    print("Triangular-solve arithmetic operations:", 2 * pairs + size)
    checks["elimination_count_" + str(size)] = bool(
        updates == size * (size - 1) * (2 * size - 1) // 6)
    checks["substitution_count_" + str(size)] = bool(
        2 * pairs + size == size ** 2)


### LU records the elimination steps
In the smaller system with rows (2, 1), (1, 3), elimination subtracts half of row 1 from row 2. The multiplier 1/2 is stored below the diagonal in L; U contains the resulting upper-triangular matrix. Here no row swaps are needed, so A=LU. For general pivoted elimination we write PA=LU.

The following cell checks the factors, solves Lc=b by forward substitution, then U theta=c by back substitution. We show the arithmetic explicitly so that the two stages are visible. In routine code, numpy.linalg.solve performs the factorisation and solves internally for square nonsingular systems.

In [ ]:
lu_A = np.array([[2., 1.], [1., 3.]])
lu_b = np.array([4., 7.])
L = np.array([[1., 0.], [0.5, 1.]])
U = np.array([[2., 1.], [0., 2.5]])
print("L (the 0.5 multiplier is recorded below the diagonal):"); print(L)
print("U:"); print(U)
print("L times U reconstructs A:", np.allclose(np.dot(L, U), lu_A))
intermediate = np.array([lu_b[0], lu_b[1] - 0.5 * lu_b[0]])
last_answer = intermediate[1] / U[1, 1]
first_answer = (intermediate[0] - U[0, 1] * last_answer) / U[0, 0]
lu_answer = np.array([first_answer, last_answer])
print("Forward substitution gives c:", intermediate)
print("Back substitution gives theta:", lu_answer)
print("Matches solve:", np.allclose(lu_answer, np.linalg.solve(lu_A, lu_b)))
checks["lu_record"] = bool(np.allclose(np.dot(L, U), lu_A))
checks["lu_two_solves"] = bool(np.allclose(np.dot(lu_A, lu_answer), lu_b))


### Underdetermined and dependent systems
The system $\theta_1+\theta_2=2$ has infinitely many exact solutions. `lstsq` chooses the one with smallest Euclidean coefficient norm, here $(1,1)$. This is an extra choice; the data alone do not identify the two coefficients. Its interpretation depends on the units/scaling of the coefficients.

Duplicating a feature also creates nonunique coefficients. For a constant input $x=1$, the model $\theta_1+\theta_2$ can have the same value for many coefficient pairs. Even a square or tall design matrix can have dependent columns.

In [ ]:
wide_A = np.array([[1., 1.]])
wide_b = np.array([2.])
minimum_norm = np.linalg.lstsq(wide_A, wide_b, rcond=None)[0]
print("Minimum-norm solution:", minimum_norm)
for candidate in [np.array([0., 2.]), minimum_norm, np.array([2., 0.])]:
    print("Coefficients:", candidate, "prediction:", np.dot(wide_A, candidate),
          "squared norm:", np.dot(candidate, candidate))
checks["minimum_norm"] = bool(np.allclose(minimum_norm, [1, 1]))
fig, axs = plt.subplots(1, 2, figsize=(10, 3.3))
u = np.linspace(-0.6, 2.6, 200)
axs[0].plot(u, 2-u, color="#006D77", label="All exact solutions")
for radius in [np.sqrt(2), 2]:
    angle = np.linspace(0, 2*np.pi, 300)
    axs[0].plot(radius*np.cos(angle), radius*np.sin(angle), color="#A7B4BA", linewidth=1)
axs[0].scatter([1], [1], color="#D56A35", label="Smallest coefficient norm")
axs[0].set_aspect("equal"); axs[0].set_xlim(-0.6,2.6); axs[0].set_ylim(-0.6,2.6)
axs[0].set_xlabel("theta_1");axs[0].set_ylabel("theta_2");axs[0].legend(fontsize=8)
axs[0].set_title("An extra rule chooses one solution")
xx = np.linspace(0,4,200)
for c0,c1 in [(2,0),(1,1),(0,2)]:axs[1].plot(xx,c0+c1*xx,label=str((c0,c1)))
axs[1].scatter([1],[2],color="#203040",zorder=5)
axs[1].set_title("Same observed input, different predictions elsewhere")
axs[1].set_xlabel("Input x");axs[1].set_ylabel("Predicted y");axs[1].legend(title="(intercept, slope)",fontsize=8)
for ax in axs:ax.grid(alpha=0.18)
fig.tight_layout();save_figure("nonunique",fig)


## 3. Regression and least squares
### Design matrix and least squares
Choose $p$ basis functions $f_1,\ldots,f_p$. They may act on a scalar input or a feature vector. The model is $\hat f(x)=\sum_j\theta_j f_j(x)$. For $N$ observations, $A_{ij}=f_j(x^{(i)})$, so $\hat y^d=A\theta$ and $r^d=y^d-A\theta$. The superscript $d$ labels the whole data vector, following the book.

For three illustrative observations $(1,1),(2,2),(3,2)$, choose $f_1(x)=1$, $f_2(x)=x$. Then $A$ has rows $(1,1),(1,2),(1,3)$. No straight line fits all three exactly. Minimise the sum of squared **vertical** residuals, not distances perpendicular to the plotted line.

In [ ]:
x = np.array([1., 2., 3.])
y = np.array([1., 2., 2.])
A = np.column_stack([np.ones(len(x)), x])
theta, solver_residuals, rank, singular_values = np.linalg.lstsq(A, y, rcond=None)
predicted = np.dot(A, theta)
residual = y - predicted
print("A:"); print(A)
print("theta:", theta)
print("predicted:", predicted)
print("observed - predicted:", residual)
print("SSE:", np.dot(residual, residual))
print("MSE:", np.mean(residual ** 2))
print("RMSE:", np.sqrt(np.mean(residual ** 2)))
print("Rank:", rank)
checks["toy_coefficients"] = bool(np.allclose(theta, [2. / 3, 0.5]))
checks["orthogonal_residual"] = bool(np.allclose(np.dot(A.T, residual), 0))
fig, ax = plt.subplots()
xx = np.linspace(0.6, 3.4, 200)
ax.plot(xx, theta[0] + theta[1] * xx, color="#006D77", label="Least-squares line")
ax.scatter(x, y, color="#203040", s=55, label="Observed")
ax.scatter(x, predicted, color="#006D77", marker="x", s=60, label="Predicted")
for i in range(3):
    ax.plot([x[i], x[i]], [predicted[i], y[i]], color="#D56A35", linewidth=3)
    ax.annotate("r" + str(i + 1), (x[i] + 0.05, (y[i] + predicted[i]) / 2), color="#A8491C")
ax.legend(fontsize=9)
finish(fig, ax, "The orange gaps are the residuals", "Input x", "Outcome y")
save_figure("residuals", fig)
fig, ax = plt.subplots(figsize=(8.8, 3.5))
b0 = np.linspace(-0.7, 2, 150); b1 = np.linspace(-0.2, 1.4, 150)
B0, B1 = np.meshgrid(b0, b1)
objective = sum((y[i] - B0 - B1 * x[i]) ** 2 for i in range(3))
contours = ax.contour(B0, B1, objective, levels=[0.2, 0.4, 0.8, 1.6, 3.2, 6.4], colors="#006D77")
ax.clabel(contours, fontsize=8)
ax.scatter([theta[0]], [theta[1]], color="#D56A35", s=70, label="Least-squares coefficients")
ax.legend(fontsize=9)
finish(fig, ax, "Each contour joins coefficients with the same squared error", "Intercept theta_1", "Slope theta_2")
save_figure("objective", fig)
metrics["toy"] = {"theta": theta.tolist(), "sse": float(np.dot(residual, residual)), "rmse": float(np.sqrt(np.mean(residual ** 2)))}


### SSE, MSE and RMSE
SSE is the **sum** of squared residuals. MSE is their **mean**, SSE/N. RMSE is sqrt(MSE), putting the error in outcome units. For the three-point example, SSE=1/6, MSE=1/18 and RMSE is about 0.236. On a fixed dataset, minimising any of these chooses the same coefficients.

### Projection and numerical computation
The least-squares objective is $\|A\theta-y^d\|_2^2$. The fitted prediction is the point in the column space of $A$ closest to $y^d$. Its residual is perpendicular to every column: $A^T(y^d-A\hat\theta)=0$. These are the normal equations. With independent columns the coefficient minimiser is unique; predictions are unique even if coefficients are not.

The formula $(A^TA)^{-1}A^Ty^d$ explains the algebra under full column rank, but is not the preferred computational recipe. Forming $A^TA$ squares the 2-norm condition number, making nearly dependent features more difficult numerically.

Conceptually, full-column-rank QR writes $A=QR$ with perpendicular unit columns in Q and a triangular R. The fitted coefficients satisfy $R\hat\theta=Q^Ty^d$. In practice we use `np.linalg.lstsq(A, y, rcond=None)[0]` without constructing these factors ourselves. QR is one solving route; the general NumPy lstsq routine uses an SVD-based least-squares solver and also handles rank deficiency. The next cell shows the explicit QR and triangular-solve route.

### QR in Python: independent columns
For a tall design matrix with independent columns, compute reduced QR, project the outcomes to Q.T y, and solve the triangular system R theta = Q.T y. No A.T A or inverse is formed. QR can be cheaper than an SVD-based solver for full-rank problems, but both have leading order O(N p^2) for tall matrices; actual speed depends on the matrix and library implementation. This explicit route requires full column rank. NumPy lstsq is the more general SVD-based option.

The following cell uses the three-observation example defined earlier and checks that QR and lstsq agree.

**What does reduced mean?** For a tall N by p matrix, reduced QR returns Q with shape (N, p) and R with shape (p, p). Complete QR would return Q with shape (N, N), including extra orthogonal directions we do not need. Both reconstruct A.

**Why solve_triangular?** R is square and nonsingular here, so numpy.linalg.solve(R, c) also works. The specialised triangular solver uses back substitution directly, with O(p^2) work. We compare their answers below, not their timings.

In [ ]:
Q,R = np.linalg.qr(A,mode="reduced")
from scipy.linalg import solve_triangular
projected_y = np.dot(Q.T, y)
theta_qr = solve_triangular(R, projected_y)
theta_general_solve = np.linalg.solve(R, projected_y)
print("General solve agrees with triangular solve:", np.allclose(theta_general_solve, theta_qr))
checks["qr_triangular_vs_general_solve"] = bool(np.allclose(theta_general_solve, theta_qr))
print("Reduced QR shapes:",Q.shape,R.shape)
print("QR coefficients:",theta_qr)
checks["qr_factorisation"] = bool(np.allclose(np.dot(Q,R),A) and np.allclose(np.dot(Q.T,Q),np.eye(2)))
checks["qr_matches_lstsq"] = bool(np.allclose(theta_qr,theta))
# Check the defining least-squares identity for an arbitrary perturbation.
delta = np.array([0.2,-0.1])
left=np.linalg.norm(y-np.dot(A,theta+delta))**2
right=np.linalg.norm(residual)**2+np.linalg.norm(np.dot(A,delta))**2
checks["least_squares_identity"] = bool(np.allclose(left,right))


### The size of the normal equations in machine learning
A has N rows (observations) and p columns (features, including any intercept). Its product A-transpose A has p by p entries, not N by N. The following examples are illustrative workloads rather than measurements of a particular deployed system: ten housing features plus an intercept; all pixels of a 64 by 64 RGB image plus an intercept; or 100,000 word-count features plus an intercept. Text features can be sparse, while their Gram matrix can be much denser.

A dense float64 number occupies eight bytes. These estimates cover only storing A-transpose A, not A, its factors, an inverse or solver workspace. Forming a dense Gram matrix costs O(Np²); factorising it costs O(p³). Avoiding an explicit inverse removes unnecessary work but does not make a huge dense least-squares problem cheap.

If A has dependent columns, A-transpose A is singular. Do not invert it: NumPy lstsq uses an SVD-based solver on the original A and still returns a minimum-norm least-squares solution. SVD means singular value decomposition; it handles dependent directions. We will study its mechanics later alongside eigenvalue decomposition. You can use lstsq now without knowing that algorithm. The pseudoinverse notation A-dagger y describes this choice; do not explicitly form a huge pseudoinverse either. If unique, meaningful coefficients are required, remove redundant features or impose a justified additional model restriction. For very large sparse problems, solvers such as SciPy LSQR work with products A v and A-transpose u without building A-transpose A; no derivation of that iterative algorithm is needed here.

In [ ]:
scale_examples = [("Housing: 10 inputs + intercept", 11),
                  ("64 by 64 RGB pixels + intercept", 64 * 64 * 3 + 1),
                  ("100,000 word features + intercept", 100001)]
for name, number_features in scale_examples:
    entries = number_features ** 2
    memory_bytes = 8 * entries
    print(name)
    print("p:", number_features, "entries in A.T A:", entries,
          "dense float64 storage in GB:", memory_bytes / 1e9)
print("Only dimensions and byte counts are calculated; no huge matrices are allocated.")
checks["gram_storage"] = bool(8 * 100001 ** 2 == 80001600008)


### SVD-based least squares
NumPy numpy.linalg.lstsq works with A itself: it does not form A.T A, and it uses an SVD-based solver even when an exact solution exists. It handles dependent columns, whereas the simple QR triangular-solve recipe above assumes independent columns.

The call returns four results: theta (coefficients), sums (squared residual sums), rank and singular values. Writing [0] selects just theta. The argument rcond=None asks NumPy to choose its default numerical-rank cutoff from floating-point precision and the matrix dimensions.

Try the identical-column design below. Its coefficients are not unique, but lstsq chooses the smallest-norm coefficient vector. Try changing one column and inspect rank and the fitted predictions. QR and SVD can give the same unique best fit, but they are different algorithms.

In [ ]:
# A deliberately dependent design: two identical columns.
dependent_A = np.array([[1., 1.], [2., 2.], [3., 3.]])
dependent_y = np.array([1., 2., 2.])
dependent_theta, unused_sums, dependent_rank, unused_singular = np.linalg.lstsq(
    dependent_A, dependent_y, rcond=None)
print("Dependent design rank:", dependent_rank)
print("Minimum-norm coefficients:", dependent_theta)
print("Fitted outcomes:", np.dot(dependent_A, dependent_theta))
checks["dependent_gram"] = bool(np.linalg.matrix_rank(np.dot(dependent_A.T, dependent_A)) == 1)
checks["dependent_stationarity"] = bool(np.allclose(
    np.dot(dependent_A.T, dependent_y-np.dot(dependent_A, dependent_theta)), 0))


In [ ]:
# A two-dimensional illustration of projection onto a one-column space.
a = np.array([2., 1.]); target = np.array([1., 3.])
coefficient = np.dot(a, target) / np.dot(a, a)
projection = a * coefficient
fig, ax = plt.subplots(figsize=(6.8,4))
ax.axhline(0,color="#B0BBC0",linewidth=0.8);ax.axvline(0,color="#B0BBC0",linewidth=0.8)
ax.plot([-1,4],[-0.5,2],color="#006D77",label="All possible fitted vectors")
for v,label,colour in [(target,"Observed vector", "#203040"),(projection,"Closest fitted vector", "#006D77")]:
    ax.annotate("",xy=v,xytext=(0,0),arrowprops={"arrowstyle":"->","color":colour,"lw":2.5})
    ax.text(v[0]+0.1,v[1]+0.1,label,color=colour,fontsize=10)
ax.plot([projection[0],target[0]],[projection[1],target[1]],color="#D56A35",lw=2,label="Residual: perpendicular")
# Right-angle marker from the two unit directions.
direction = a / np.linalg.norm(a); normal=(target-projection)/np.linalg.norm(target-projection)
corner=np.array([projection,projection+0.18*direction,projection+0.18*(direction+normal),projection+0.18*normal])
ax.plot(corner[:,0],corner[:,1],color="#D56A35")
ax.set_xlim(-0.5,4.4);ax.set_ylim(-0.5,4);ax.set_aspect("equal");ax.legend(loc="upper left",fontsize=8)
finish(fig,ax,"Projection in observation space", "First observation coordinate", "Second observation coordinate")
save_figure("projection",fig)

# A second view adds any other candidate fit to make Pythagoras visible.
candidate_fit = 1.8 * a
fig, ax = plt.subplots(figsize=(8.8, 3.7))
ax.fill([target[0], projection[0], candidate_fit[0]],
        [target[1], projection[1], candidate_fit[1]], color="#EDF3F5", zorder=1)
ax.plot([-0.3, 4.8], [-0.15, 2.4], color="#006D77", linewidth=1.5)
ax.plot([projection[0], target[0]], [projection[1], target[1]], color="#D56A35", linewidth=3)
ax.plot([target[0], candidate_fit[0]], [target[1], candidate_fit[1]], color="#64717B", linewidth=2)
ax.plot([projection[0], candidate_fit[0]], [projection[1], candidate_fit[1]], color="#006D77", linewidth=3)
for point in [target, projection, candidate_fit]:
    ax.scatter([point[0]], [point[1]], color="#203040", s=40, zorder=5)
ax.text(target[0]-0.35, target[1]+0.25, r"Observed $y^d$", fontsize=13)
ax.text(projection[0]-0.55, projection[1]-0.48, r"Closest fit $\hat y^d$", fontsize=13, color="#006D77")
ax.text(candidate_fit[0]+0.12, candidate_fit[1]-0.05, r"Other fit $z$", fontsize=13)
ax.text(1.05, 1.85, r"$r^d$", fontsize=15, color="#D56A35")
ax.text(2.55, 0.98, r"$z-\hat y^d$", fontsize=13, color="#006D77")
ax.text(2.65, 2.75, r"$y^d-z$", fontsize=13, color="#64717B")
ax.text(3.5, 2.42, r"Fitted space: $\mathrm{col}(A)$", fontsize=12, color="#006D77")
unit_direction = a / np.linalg.norm(a)
unit_normal = (target-projection) / np.linalg.norm(target-projection)
right_angle = np.array([projection+0.18*unit_direction,
                        projection+0.18*(unit_direction+unit_normal),
                        projection+0.18*unit_normal])
ax.plot(right_angle[:, 0], right_angle[:, 1], color="#D56A35", linewidth=1.5)
ax.set_xlim(-0.4, 5.5); ax.set_ylim(-0.2, 3.8); ax.set_aspect("equal"); ax.axis("off")
fig.tight_layout(); save_figure("pythagoras", fig)
checks["projection_pythagoras"] = bool(np.allclose(
    np.linalg.norm(target-candidate_fit)**2,
    np.linalg.norm(target-projection)**2+np.linalg.norm(candidate_fit-projection)**2))


### Similar features and sensitive coefficients
The two columns of A below are almost identical. Compare two outcome vectors that differ by at most 0.001. Fit each using the same matrix and lstsq. The coefficients move from (1, 1) to (0, 2), despite the tiny change in outcomes. Both fits satisfy their equations to rounding precision. This illustrates sensitivity of the coefficients, without introducing a condition-number formula.

In [ ]:
similar_A = np.array([[1., 1.], [1., 1.001], [1., 0.999]])
similar_b = np.array([2., 2.001, 1.999])
changed_b = np.array([2., 2.002, 1.998])
similar_theta = np.linalg.lstsq(similar_A, similar_b, rcond=None)[0]
changed_theta = np.linalg.lstsq(similar_A, changed_b, rcond=None)[0]
print("A:", similar_A, sep="\n")
print("Original outcomes:", similar_b)
print("Changed outcomes:", changed_b)
print("Largest outcome change:", np.max(np.abs(changed_b - similar_b)))
print("Original coefficients:", np.round(similar_theta, 6))
print("Changed coefficients:", np.round(changed_theta, 6))
print("Original fit matches outcomes:", np.allclose(np.dot(similar_A, similar_theta), similar_b))
print("Changed fit matches outcomes:", np.allclose(np.dot(similar_A, changed_theta), changed_b))
checks["similar_features_original"] = bool(np.allclose(similar_theta, [1., 1.]))
checks["similar_features_changed"] = bool(np.allclose(changed_theta, [0., 2.]))
checks["similar_features_small_change"] = bool(np.max(np.abs(changed_b-similar_b)) < 0.00101)
fig, ax = plt.subplots(figsize=(5.5, 3.3))
positions = np.arange(2); width = 0.32
ax.bar(positions-width/2, similar_theta, width, label="Original outcomes", color="#006D77")
ax.bar(positions+width/2, changed_theta, width, label="Outcomes changed by at most 0.001", color="#D56A35")
ax.set_xticks(positions); ax.set_xticklabels(["Coefficient 1", "Coefficient 2"])
ax.set_ylabel("Fitted coefficient"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.18)
fig.tight_layout(); save_figure("conditioning", fig)


## 4. Polynomial regression and model selection
Changing basis functions changes the model, not the fitting algorithm. A degree-$d$ polynomial uses $1,x,\ldots,x^d$ and has $p=d+1$ coefficients. It is nonlinear in the input but linear in its coefficients.

We generate illustrative data from a quadratic plus measurement error, with fixed seed 587. This construction is known to us, but hidden when choosing models. Training determines coefficients; a separate validation set chooses the degree; a third test set is used once for the final score. Lower training error alone does not justify a more flexible model. Under comparable validation performance, prefer the simpler model (Occam's razor).

In this particular sample, the strict minimum validation RMSE is at degree 4 (about 0.231). Degrees 2 (about 0.234) and 3 (about 0.233) are nearly as good. The code below reports the strict-minimum rule; it is not evidence that degree 4 is meaningfully better. Degree 2 is a defensible simpler choice, and the final test set should not be used to make this choice.

In [ ]:
rng=np.random.RandomState(587)
poly_x=np.linspace(-1,1,15)
def underlying_function(values):
    return 0.6+0.8*values-1.2*values**2
poly_y=underlying_function(poly_x)+0.22*rng.randn(len(poly_x))
val_x=rng.uniform(-1,1,100);val_y=underlying_function(val_x)+0.22*rng.randn(100)
test_x=rng.uniform(-1,1,100);test_y=underlying_function(test_x)+0.22*rng.randn(100)
def polynomial_matrix(values,degree):
    return np.column_stack([values**j for j in range(degree+1)])
poly_grid=np.linspace(-1.15,1.15,400)
fig,axs=plt.subplots(1,3,figsize=(11,3.2),sharey=True)
for ax,degree in zip(axs,[1,2,12]):
    P=polynomial_matrix(poly_x,degree)
    coefficients=np.linalg.lstsq(P,poly_y,rcond=None)[0]
    ax.scatter(poly_x,poly_y,color="#203040",s=25)
    ax.plot(poly_grid,np.dot(polynomial_matrix(poly_grid,degree),coefficients),color="#006D77")
    ax.set_title("Degree "+str(degree));ax.set_xlabel("x");ax.set_ylim(-2,2);ax.axvspan(-1.15,-1,color="#E9EDEF");ax.axvspan(1,1.15,color="#E9EDEF");ax.grid(alpha=0.18)
axs[0].set_ylabel("Outcome y")
fig.suptitle("Same training points, different model families (illustrative)",fontsize=12)
fig.tight_layout();save_figure("polynomials",fig)
training_errors=[];validation_errors=[];poly_coefficients=[]
for degree in range(13):
    P=polynomial_matrix(poly_x,degree)
    coefficients=np.linalg.lstsq(P,poly_y,rcond=None)[0]
    poly_coefficients.append(coefficients)
    training_errors.append(np.sqrt(np.mean((poly_y-np.dot(P,coefficients))**2)))
    validation_errors.append(np.sqrt(np.mean((val_y-np.dot(polynomial_matrix(val_x,degree),coefficients))**2)))
selected_degree=int(np.argmin(validation_errors))
selected_poly=poly_coefficients[selected_degree]
final_poly_rmse=float(np.sqrt(np.mean((test_y-np.dot(polynomial_matrix(test_x,selected_degree),selected_poly))**2)))
fig,ax=plt.subplots()
ax.plot(range(13),training_errors,"o-",label="Training RMSE",color="#006D77")
ax.plot(range(13),validation_errors,"o-",label="Validation RMSE",color="#D56A35")
ax.axvline(selected_degree,color="#64717B",linestyle="--",label="Lowest validation RMSE: degree 4")
ax.legend(fontsize=9)
finish(fig,ax,"Training error is not the model-selection rule", "Polynomial degree", "RMSE")
save_figure("validation",fig)
print("Degree with strictly lowest validation RMSE:",selected_degree)
for degree in [2, 3, 4]:
    print("Degree", degree, "validation RMSE:", round(validation_errors[degree], 4))
print("Degrees 2 and 3 are nearly as good; degree 2 is a reasonable simpler choice.")
print("Test RMSE, checked once:",final_poly_rmse)
metrics["polynomial"]={"selected_degree":selected_degree,"test_rmse":final_poly_rmse,"training_rmse":training_errors,"validation_rmse":validation_errors}
checks["training_error_nonincreasing"] = bool(np.all(np.diff(training_errors)<1e-8))
checks["poly_full_rank"] = bool(np.linalg.matrix_rank(polynomial_matrix(poly_x,12))==13)


## 5. Time-series regression
The input can be time. Here monthly Odense temperatures are fitted with a constant, a linear trend, and a repeating annual sine/cosine cycle. `tau` measures years from January 2006. The basis is chosen before fitting, with no future temperatures as inputs.

Keep chronology: train on 2006–2023, use 2024 to select among constant / seasonal / seasonal-plus-trend models, then refit the selected family using 2006–2024 and score 2025 once. Each forecast value is a **monthly mean**. This differs from Class 4's daily ARMA example. The trend is an empirical local fitting term, not evidence of causation or a global climate trend.

In [ ]:
weather=pd.read_csv("odense-weather.csv",parse_dates=["date"]).set_index("date").sort_index()
monthly=weather["temp_mean"].resample("MS").mean()
tau=np.arange(len(monthly),dtype=float)/12
def time_matrix(t,kind):
    ones=np.ones(len(t))
    if kind=="constant":return ones.reshape(-1,1)
    if kind=="seasonal":return np.column_stack([ones,np.sin(2*np.pi*t),np.cos(2*np.pi*t)])
    return np.column_stack([ones,t,np.sin(2*np.pi*t),np.cos(2*np.pi*t)])
train_time=monthly.index.year<=2023
validate_time=monthly.index.year==2024
test_time=monthly.index.year==2025
candidates=["constant","seasonal","seasonal + trend"]
time_scores=[]
for kind in candidates:
    coefficients=np.linalg.lstsq(time_matrix(tau[train_time],kind),monthly.values[train_time],rcond=None)[0]
    validation_prediction=np.dot(time_matrix(tau[validate_time],kind),coefficients)
    time_scores.append(float(np.sqrt(np.mean((monthly.values[validate_time]-validation_prediction)**2))))
selected_kind=candidates[int(np.argmin(time_scores))]
print("Validation RMSE (2024):")
for kind,score in zip(candidates,time_scores):print(kind,round(score,3))
fit_time=~test_time
final_time_A=time_matrix(tau[fit_time],selected_kind)
final_time_coefficients=np.linalg.lstsq(final_time_A,monthly.values[fit_time],rcond=None)[0]
time_prediction=np.dot(time_matrix(tau[test_time],selected_kind),final_time_coefficients)
time_rmse=float(np.sqrt(np.mean((monthly.values[test_time]-time_prediction)**2)))
print("Selected model:",selected_kind)
print("Final coefficients:",final_time_coefficients)
print("Test RMSE (2025), checked once:",round(time_rmse,3))
fig,ax=plt.subplots(figsize=(9,3.6))
show=monthly.index.year>=2021
ax.plot(monthly.index[show],monthly.values[show],"o-",markersize=3,color="#203040",label="Observed monthly means")
all_predictions=np.dot(time_matrix(tau,selected_kind),final_time_coefficients)
ax.plot(monthly.index[show & fit_time],all_predictions[show & fit_time],color="#006D77",label="Fitted through 2024")
ax.plot(monthly.index[test_time],time_prediction,"o--",markersize=4,color="#D56A35",label="2025 forecast: unseen during selection")
ax.axvline(pd.Timestamp("2025-01-01"),color="#64717B",linestyle="--")
ax.legend(fontsize=8,ncol=2)
finish(fig,ax,"A regression model can use time as its input", "Date", "Monthly mean temperature (degrees C)")
save_figure("time-series",fig)
metrics["time_series"]={"candidates":candidates,"validation_rmse":time_scores,"selected":selected_kind,"test_rmse":time_rmse,"coefficients":final_time_coefficients.tolist()}
checks["time_split"] = bool(train_time.sum()==216 and validate_time.sum()==12 and test_time.sum()==12 and fit_time.sum()==228)
checks["time_forecast_finite"] = bool(np.isfinite(time_prediction).all())
checks["time_design_full_rank"] = bool(np.linalg.matrix_rank(final_time_A)==final_time_A.shape[1])


### Past values can also be features
An autoregression with two lags predicts $\hat T_{t+1}=\theta_1+\theta_2T_t+\theta_3T_{t-1}$. One design row is `[1, T_t, T_(t-1)]`, and its target is `T_(t+1)`. Fitting is still least squares. This uses observed lagged values in training; a multi-step forecast would reuse earlier predictions when future lagged inputs are unavailable. Unlike ARMA, this AR model has no moving-average error term.

The example below only constructs training rows so the input/target alignment is explicit.

In [ ]:
recent=np.array([4.,5.,7.,6.,8.])
lag_A=np.column_stack([np.ones(len(recent)-2),recent[1:-1],recent[:-2]])
lag_y=recent[2:]
print("Columns: constant, today's value, yesterday's value")
print(lag_A)
print("Targets: next value")
print(lag_y)
checks["lag_alignment"] = bool(np.allclose(lag_A[0],[1,5,4]) and lag_y[0]==7)


## Try changing the model
- Change a housing observation and inspect the line and its residuals.
- Add a duplicate column; inspect rank, coefficients and predictions.
- Change polynomial degree while keeping training and validation data fixed.
- Try a new time-series cutoff, keeping later dates out of fitting and model selection.

Use `solve` for a nonsingular square system, and `lstsq` for least squares and minimum-norm solutions. Always inspect units, dimensions, rank and residuals. Full-rank QR explains the computation; rank-deficient problems need an extra convention for the coefficients.

Sources: [Boyd and Vandenberghe](https://web.stanford.edu/~boyd/vmls/), [NumPy solve](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html), [NumPy lstsq](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html), [NumPy QR](https://numpy.org/doc/stable/reference/generated/numpy.linalg.qr.html), [SciPy triangular solve](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_triangular.html). Weather: ERA5 via Open-Meteo, CC BY 4.0; see DATA-SOURCE.md in the bundle.

## Optional: several right-hand sides
Our regression has one outcome vector. If the coefficient matrix stays the same while the required outcome changes, put those known vectors in columns of B and solve M X = B. This optional extension uses the earlier 3-by-3 matrix M.

In [ ]:
# Several right-hand sides in one call: the solver can reuse its factorisation.
B = np.column_stack([b, [1., 0., 0.]])
X = np.linalg.solve(M, B)
checks["multiple_rhs"] = bool(np.allclose(np.dot(M, X), B))
print("Two right-hand sides, solution columns:"); print(X)
